# J6 — Analyse de la concurrence dans la restauration

Objectifs :

- charger la BPE 2025 détaillée ;
- sélectionner les restaurants franciliens ;
- distinguer restauration rapide, traditionnelle et cafétérias ;
- agréger les établissements par commune ;
- calculer les densités concurrentielles ;
- joindre les résultats au profil communal ;
- contrôler les communes fusionnées et les codes géographiques.

In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

from pathlib import Path
from zipfile import ZipFile
import csv
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

print("Pandas :", pd.__version__)
print("NumPy :", np.__version__)
print("OMP_NUM_THREADS :", os.environ["OMP_NUM_THREADS"])
print("NUMEXPR_NUM_THREADS :", os.environ["NUMEXPR_NUM_THREADS"])
print("Bibliothèques chargées correctement ✅")

Pandas : 2.2.2
NumPy : 1.26.4
OMP_NUM_THREADS : 1
NUMEXPR_NUM_THREADS : 1
Bibliothèques chargées correctement ✅


In [2]:
#Retrouver le projet
def trouver_racine_projet():
    candidats = [Path.cwd(), *Path.cwd().parents]

    for dossier in candidats:
        if (dossier / "data").exists():
            return dossier

    raise FileNotFoundError(
        "Dossier data introuvable. Lance le notebook depuis le projet GeoMarketing IDF."
    )


RACINE = trouver_racine_projet()  / "OneDrive/Geomarketing_IDF"

DOSSIER_RAW = RACINE  / "data" / "raw"
DOSSIER_BPE = DOSSIER_RAW / "bpe"
DOSSIER_PROCESSED = RACINE / "data" / "processed"

DOSSIER_BPE.mkdir(parents=True, exist_ok=True)
DOSSIER_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Racine du projet :", RACINE)
print("Dossier BPE :", DOSSIER_BPE)
print("Dossier de sortie :", DOSSIER_PROCESSED)

Racine du projet : C:\Users\almou\OneDrive\Geomarketing_IDF
Dossier BPE : C:\Users\almou\OneDrive\Geomarketing_IDF\data\raw\bpe
Dossier de sortie : C:\Users\almou\OneDrive\Geomarketing_IDF\data\processed


In [3]:
noms_profils_possibles = [
    "profil_communes_idf_j5.csv",
    "profil_communes_idf.csv",
    "profil_socioeconomique_idf.csv",
    "profil_final_idf.csv",
]

FICHIER_PROFIL = None

for nom in noms_profils_possibles:
    candidats = list(RACINE.rglob(nom))

    if candidats:
        FICHIER_PROFIL = candidats[0]
        break

if FICHIER_PROFIL is None:
    raise FileNotFoundError(
        "Profil communal introuvable. Vérifie les fichiers du dossier data/processed."
    )

print("Profil communal détecté :", FICHIER_PROFIL)

Profil communal détecté : C:\Users\almou\OneDrive\Geomarketing_IDF\data\processed\insee\profil_socioeconomique_idf.csv


In [4]:
extensions_acceptees = {".csv", ".parquet", ".zip"}

candidats_bpe = []

for fichier in DOSSIER_BPE.rglob("*"):
    nom = fichier.name.lower()

    if (
        fichier.is_file()
        and fichier.suffix.lower() in extensions_acceptees
        and "bpe" in nom
        and not any(
            mot in nom
            for mot in ["dictionnaire", "liste", "passage", "source", "documentation"]
        )
    ):
        candidats_bpe.append(fichier)

if not candidats_bpe:
    raise FileNotFoundError(
        "Aucun fichier BPE trouvé dans data/raw/bpe. "
        "Télécharge le fichier détail BPE 2025."
    )

# Le fichier principal est normalement le plus volumineux.
FICHIER_BPE = max(candidats_bpe, key=lambda fichier: fichier.stat().st_size)

print("BPE détectée :", FICHIER_BPE)
print("Taille :", round(FICHIER_BPE.stat().st_size / 1_000_000, 1), "Mo")

BPE détectée : C:\Users\almou\OneDrive\Geomarketing_IDF\data\raw\bpe\BPE25.csv
Taille : 1515.3 Mo


In [5]:
def detecter_separateur(fichier):
    with open(fichier, "r", encoding="utf-8-sig", errors="replace") as flux:
        premiere_ligne = flux.readline()

    return ";" if premiere_ligne.count(";") > premiere_ligne.count(",") else ","


def lire_fichier(fichier):
    suffixe = fichier.suffix.lower()

    if suffixe == ".parquet":
        return pd.read_parquet(fichier)

    if suffixe == ".csv":
        separateur = detecter_separateur(fichier)

        return pd.read_csv(
            fichier,
            sep=separateur,
            dtype=str,
            encoding="utf-8-sig",
            low_memory=False,
        )

    if suffixe == ".zip":
        with ZipFile(fichier) as archive:
            membres_csv = [
                information
                for information in archive.infolist()
                if information.filename.lower().endswith(".csv")
            ]

            if not membres_csv:
                raise ValueError("Aucun CSV trouvé dans l'archive BPE.")

            membre = max(membres_csv, key=lambda information: information.file_size)

            with archive.open(membre.filename) as flux:
                premiere_ligne = flux.readline().decode(
                    "utf-8-sig", errors="replace"
                )

            separateur = (
                ";" if premiere_ligne.count(";") > premiere_ligne.count(",") else ","
            )

            with archive.open(membre.filename) as flux:
                return pd.read_csv(
                    flux,
                    sep=separateur,
                    dtype=str,
                    encoding="utf-8-sig",
                    low_memory=False,
                )

    raise ValueError(f"Format non pris en charge : {suffixe}")

In [11]:
COLONNES_BPE_UTILES = {
    "DEPCOM",
    "TYPEQU",
    "TYPERESTO",
    "SIRET",
    "NOMRS",
    "CNOMRS",
    "NUMVOIE",
    "TYPVOIE",
    "LIBVOIE",
    "CODPOS",
    "APET",
    "LATITUDE",
    "LONGITUDE",
}

separateur = detecter_separateur(FICHIER_BPE)

morceaux_restaurants = []

lecteur = pd.read_csv(
    FICHIER_BPE,
    sep=separateur,
    usecols=lambda colonne: (
        str(colonne)
        .replace("\ufeff", "")
        .strip()
        .upper()
        in COLONNES_BPE_UTILES
    ),
    dtype=str,
    encoding="utf-8-sig",
    encoding_errors="replace",
    chunksize=100_000,
    low_memory=True,
)

for numero, morceau in enumerate(lecteur, start=1):

    morceau.columns = [
        str(colonne)
        .replace("\ufeff", "")
        .strip()
        .upper()
        for colonne in morceau.columns
    ]

    if "TYPEQU" not in morceau.columns:
        raise ValueError(
            "TYPEQU est absente : vérifie que tu utilises "
            "le fichier détaillé géolocalisé de 136 Mo."
        )

    # Conserver immédiatement uniquement les restaurants A504
    morceau = morceau[
        morceau["TYPEQU"]
        .astype("string")
        .str.strip()
        .str.upper()
        .eq("A504")
    ].copy()

    if not morceau.empty:
        morceaux_restaurants.append(morceau)

    print(
        f"Morceau {numero} traité — "
        f"{len(morceau)} restaurants conservés"
    )

if not morceaux_restaurants:
    raise ValueError(
        "Aucun restaurant A504 n’a été trouvé dans le fichier."
    )

bpe = pd.concat(
    morceaux_restaurants,
    ignore_index=True,
)

print("Dimensions après filtrage :", bpe.shape)
print("Colonnes chargées :", bpe.columns.tolist())
print("BPE restauration chargée sans saturer la mémoire ✅")

profil = pd.read_csv(
    FICHIER_PROFIL,
    sep=None,
    engine="python",
    dtype=str,
    encoding="utf-8-sig",
)

print("Dimensions BPE :", bpe.shape)
print("Dimensions profil :", profil.shape)

display(bpe.head())
display(profil.head())

Morceau 1 traité — 11842 restaurants conservés
Morceau 2 traité — 15877 restaurants conservés
Morceau 3 traité — 8097 restaurants conservés
Morceau 4 traité — 7036 restaurants conservés
Morceau 5 traité — 8754 restaurants conservés
Morceau 6 traité — 7029 restaurants conservés
Morceau 7 traité — 6213 restaurants conservés
Morceau 8 traité — 6273 restaurants conservés
Morceau 9 traité — 6570 restaurants conservés
Morceau 10 traité — 6753 restaurants conservés
Morceau 11 traité — 7552 restaurants conservés
Morceau 12 traité — 6187 restaurants conservés
Morceau 13 traité — 6502 restaurants conservés
Morceau 14 traité — 8224 restaurants conservés
Morceau 15 traité — 5508 restaurants conservés
Morceau 16 traité — 7565 restaurants conservés
Morceau 17 traité — 6947 restaurants conservés
Morceau 18 traité — 7520 restaurants conservés
Morceau 19 traité — 7034 restaurants conservés
Morceau 20 traité — 7633 restaurants conservés
Morceau 21 traité — 7600 restaurants conservés
Morceau 22 traité — 

,APET,NOMRS,CNOMRS,NUMVOIE,TYPVOIE,LIBVOIE,CODPOS,DEPCOM,TYPEQU,SIRET,TYPERESTO,LONGITUDE,LATITUDE
0,5610A,BAIT CAROLE,NaN,NaN,CRS,FORBIN CARENAGE,97139,97101,A504,43801090200015,5610A,-61.52965,16.22828
1,5610C,SARL KARAK CENTRE,NaN,NaN,LD,CARAQUE,97139,97101,A504,34259068400011,5610C,-61.49362,16.25523
2,5610C,CLAUDEON HILAIRE,NaN,NaN,LD,GRAND CAMP,97139,97101,A504,38475401600028,5610C,-61.53696,16.25092
3,5610A,CARIEN FRANCETTE,NaN,NaN,LD,MORNE CARUEL,97139,97101,A504,38483120200013,5610A,-61.51994,16.2553
4,5610A,JACOBY-KOALY SERVAIS,NaN,NaN,LD,PROVIDENCE,97139,97101,A504,38077579100011,5610A,-61.51071,16.26481


,CODGEO,NOM_COMMUNE,DEP,REG,POPULATION_2011,POPULATION_2016,POPULATION_2022,POP_0_14_ANS_2022,POP_15_29_ANS_2022,POP_30_44_ANS_2022,POP_45_59_ANS_2022,POP_60_74_ANS_2022,POP_75_89_ANS_2022,POP_90_ANS_PLUS_2022,POP_MOINS_30_ANS_2022,POP_15_44_ANS_2022,POP_60_ANS_PLUS_2022,POP_75_ANS_PLUS_2022,PART_0_14_ANS_PCT,PART_15_29_ANS_PCT,PART_30_44_ANS_PCT,PART_MOINS_30_ANS_PCT,PART_15_44_ANS_PCT,PART_60_ANS_PLUS_PCT,PART_75_ANS_PLUS_PCT,EVOLUTION_POP_2016_2022,EVOLUTION_POP_2016_2022_PCT,TAUX_ANNUEL_POP_2016_2022_PCT,REVENU_MEDIAN_EUROS,TAUX_PAUVRETE_PCT,EMPLOIS_SALARIES_LIEU_TRAVAIL,NOMBRE_ETABLISSEMENTS,EMPLOIS_SALARIES_POUR_100_HAB,ETABLISSEMENTS_POUR_1000_HAB
0,75056,Paris,75,11,2249975.0,2190327.0,2113705.0,274280.561508206,513386.75602884,457511.658563684,387099.215350194,302819.494109117,154040.332161959,24566.9822779995,787667.317537046,970898.4145925241,481426.8085490755,178607.31443995849,12.98,24.29,21.65,37.26,45.93,22.78,8.45,-76622.0,-3.5,-0.59,33650.0,16.8,1551133.75022,188002.0,73.38459010221389,88.94429449710343
1,77001,Achères-la-Forêt,77,11,1232.0,1139.0,1183.0,182.135094788161,159.522993092748,189.869837113436,347.331906368833,204.113784168138,89.442063221765,10.5843212469189,341.658087880909,349.39283020618404,304.1401686368219,100.0263844686839,15.4,13.48,16.05,28.88,29.53,25.71,8.46,44.0,3.86,0.63,34840.0,NaN,130.30679,35.0,11.014944209636518,29.585798816568047
2,77002,Amillis,77,11,781.0,819.0,821.0,127.936213516012,122.351987081097,146.913593456989,151.60336129515,188.399316742963,59.3206016148823,24.4749262929075,250.28820059710898,269.265580538086,272.1948446507528,83.7955279077898,15.58,14.9,17.89,30.49,32.8,33.15,10.21,2.0,0.24,0.04,29130.0,NaN,190.69785,39.0,23.227509135200975,47.503045066991476
3,77003,Amponville,77,11,388.0,351.0,351.0,63.0,38.0,72.0,88.0,62.0,28.0,0.0,101.0,110.0,90.0,28.0,17.95,10.83,20.51,28.77,31.34,25.64,7.98,0.0,0.0,0.0,30310.0,NaN,19.95946,12.0,5.6864558404558405,34.18803418803419
4,77004,Andrezel,77,11,299.0,285.0,324.0,77.0,33.0,78.0,53.0,67.0,15.0,1.0,110.0,111.0,83.0,16.0,23.77,10.19,24.07,33.95,34.26,25.62,4.94,39.0,13.68,2.16,31970.0,NaN,23.16751,13.0,7.150466049382716,40.123456790123456


In [12]:
def normaliser_nom_colonne(nom):
    nom = str(nom).strip().upper()

    nom = unicodedata.normalize("NFKD", nom)
    nom = "".join(caractere for caractere in nom if not unicodedata.combining(caractere))

    nom = re.sub(r"[^A-Z0-9]+", "_", nom)
    nom = nom.strip("_")

    return nom


bpe.columns = [normaliser_nom_colonne(colonne) for colonne in bpe.columns]
profil.columns = [normaliser_nom_colonne(colonne) for colonne in profil.columns]

print("Colonnes BPE :")
print(bpe.columns.tolist())

Colonnes BPE :
['APET', 'NOMRS', 'CNOMRS', 'NUMVOIE', 'TYPVOIE', 'LIBVOIE', 'CODPOS', 'DEPCOM', 'TYPEQU', 'SIRET', 'TYPERESTO', 'LONGITUDE', 'LATITUDE']


In [16]:

bpe.columns = [
    normaliser_nom_colonne(colonne)
    for colonne in bpe.columns
]

print("Fichier chargé :", FICHIER_BPE)
print("Dimensions :", bpe.shape)
print("Colonnes présentes :", bpe.columns.tolist())

colonnes_attendues = {
    "DEPCOM",
    "TYPEQU",
    "TYPERESTO",
    "LATITUDE",
    "LONGITUDE",
}

print(
    "Colonnes attendues présentes :",
    colonnes_attendues.issubset(bpe.columns)
)

for colonne in bpe.columns:
    print(repr(colonne))

colonnes_bpe_obligatoires = {"DEPCOM", "TYPEQU"}

colonnes_absentes = colonnes_bpe_obligatoires - set(bpe.columns)

if colonnes_absentes:
    raise ValueError(
        f"Colonnes BPE absentes : {colonnes_absentes}. "
        "Vérifie que tu as téléchargé le fichier détail BPE 2025."
    )

if "TYPERESTO" not in bpe.columns:
    raise ValueError(
        "La colonne TYPERESTO est absente. "
        "Tu utilises probablement une ancienne version de la BPE. "
        "Télécharge la BPE 2025 détaillée."
    )

if "CODGEO" not in profil.columns:
    candidats_code = ["DEPCOM", "CODE_COMMUNE", "COM"]

    colonne_code = next(
        (colonne for colonne in candidats_code if colonne in profil.columns),
        None,
    )

    if colonne_code is None:
        raise ValueError("Aucune colonne de code commune trouvée dans le profil.")

    profil = profil.rename(columns={colonne_code: "CODGEO"})

print("Colonnes obligatoires présentes ✅")

Fichier chargé : C:\Users\almou\OneDrive\Geomarketing_IDF\data\raw\bpe\BPE25.csv
Dimensions : (231989, 13)
Colonnes présentes : ['APET', 'NOMRS', 'CNOMRS', 'NUMVOIE', 'TYPVOIE', 'LIBVOIE', 'CODPOS', 'DEPCOM', 'TYPEQU', 'SIRET', 'TYPERESTO', 'LONGITUDE', 'LATITUDE']
Colonnes attendues présentes : True
'APET'
'NOMRS'
'CNOMRS'
'NUMVOIE'
'TYPVOIE'
'LIBVOIE'
'CODPOS'
'DEPCOM'
'TYPEQU'
'SIRET'
'TYPERESTO'
'LONGITUDE'
'LATITUDE'
Colonnes obligatoires présentes ✅


In [17]:
def normaliser_code_commune(serie):
    return (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.upper()
        .str.zfill(5)
    )


bpe["DEPCOM"] = normaliser_code_commune(bpe["DEPCOM"])
profil["CODGEO"] = normaliser_code_commune(profil["CODGEO"])

assert profil["CODGEO"].notna().all()
assert profil["CODGEO"].is_unique
assert profil["CODGEO"].str.fullmatch(r"\d{5}").all()

print("Codes communaux normalisés ✅")

Codes communaux normalisés ✅


In [18]:
DEPARTEMENTS_IDF = {
    "75",
    "77",
    "78",
    "91",
    "92",
    "93",
    "94",
    "95",
}

bpe_idf = bpe[
    bpe["DEPCOM"].str[:2].isin(DEPARTEMENTS_IDF)
].copy()

print("Équipements en Île-de-France :", len(bpe_idf))
print(bpe_idf["DEPCOM"].nunique(), "codes géographiques représentés")

Équipements en Île-de-France : 49169
951 codes géographiques représentés


In [19]:
#Filtrer sur l'Ile-de-France

masque_paris = bpe_idf["DEPCOM"].str.fullmatch(r"751\d{2}", na=False)

bpe_idf["CODGEO"] = bpe_idf["DEPCOM"]
bpe_idf.loc[masque_paris, "CODGEO"] = "75056"

print("Équipements situés dans les arrondissements parisiens :", masque_paris.sum())
print("Code utilisé après regroupement : 75056")

Équipements situés dans les arrondissements parisiens : 20911
Code utilisé après regroupement : 75056


In [20]:
#Sélectionner les restaurants
restaurants = bpe_idf[
    bpe_idf["TYPEQU"].astype("string").str.strip().str.upper().eq("A504")
].copy()

restaurants["TYPERESTO"] = (
    restaurants["TYPERESTO"]
    .astype("string")
    .str.strip()
    .str.upper()
)

print("Restaurants franciliens détectés :", len(restaurants))
display(restaurants.head())

Restaurants franciliens détectés : 49169


,APET,NOMRS,CNOMRS,NUMVOIE,TYPVOIE,LIBVOIE,CODPOS,DEPCOM,TYPEQU,SIRET,TYPERESTO,LONGITUDE,LATITUDE,CODGEO
9998,5610C,BOULANGERIES PAUL,NaN,25,AV,DE L OPERA,75001,75101,A504,40305211101212,5610C,2.33377,48.86672,75056
9999,5610C,CAFE SIRENE FRANCE,STARBUCKS COFFEE,26,AV,DE L OPERA,75001,75101,A504,44533010300026,5610C,2.33407,48.86708,75056
10000,5610A,LE VICTORIA CAFE,NaN,25,AV,VICTORIA,75001,75101,A504,44444986200010,5610A,2.34623,48.85779,75056
10001,5610A,MANALOU,NaN,7,BD,DE LA MADELEINE,75001,75101,A504,47761841700018,5610A,2.32706,48.86959,75056
10002,5610C,CAFE SIRENE FRANCE,STARBUCKS COFFEE,11,BD,DE SEBASTOPOL,75001,75101,A504,44533010300224,5610C,2.34819,48.8589,75056


In [21]:
#Classer les restaurants
correspondance_restauration = {
    "5610A": "TRADITIONNELLE",
    "5610B": "CAFETERIA_LIBRE_SERVICE",
    "5610C": "RAPIDE",
}

restaurants["CATEGORIE_RESTAURATION"] = (
    restaurants["TYPERESTO"]
    .map(correspondance_restauration)
    .fillna("TYPE_INCONNU")
)

controle_categories = (
    restaurants["CATEGORIE_RESTAURATION"]
    .value_counts(dropna=False)
    .rename_axis("CATEGORIE")
    .reset_index(name="NOMBRE")
)

display(controle_categories)

,CATEGORIE,NOMBRE
0,TRADITIONNELLE,24764
1,RAPIDE,24319
2,CAFETERIA_LIBRE_SERVICE,86


In [22]:
#Contrôler les SIRET et doublons

if "SIRET" in restaurants.columns:
    restaurants["SIRET"] = (
        restaurants["SIRET"]
        .astype("string")
        .str.replace(r"\D", "", regex=True)
        .str.zfill(14)
    )

    siret_valide = restaurants["SIRET"].str.fullmatch(r"\d{14}", na=False)

    print("SIRET valides :", siret_valide.sum())
    print("SIRET absents ou non diffusés :", (~siret_valide).sum())

    restaurants["CLE_EQUIPEMENT"] = np.where(
        siret_valide,
        "SIRET_" + restaurants["SIRET"].fillna(""),
        "LIGNE_" + restaurants.index.astype(str),
    )

    nombre_avant = len(restaurants)

    restaurants = restaurants.drop_duplicates(
        subset=["CLE_EQUIPEMENT"],
        keep="first",
    ).copy()

    print("Doublons SIRET supprimés :", nombre_avant - len(restaurants))

else:
    restaurants["CLE_EQUIPEMENT"] = (
        "LIGNE_" + restaurants.index.astype(str)
    )

    print("Colonne SIRET absente : aucune déduplication par SIRET.")

SIRET valides : 49169
SIRET absents ou non diffusés : 0
Doublons SIRET supprimés : 0


In [23]:
#Contrôler les coordonnées

for colonne in ["LATITUDE", "LONGITUDE"]:
    if colonne in restaurants.columns:
        restaurants[colonne] = pd.to_numeric(
            restaurants[colonne].astype(str).str.replace(",", ".", regex=False),
            errors="coerce",
        )

if {"LATITUDE", "LONGITUDE"}.issubset(restaurants.columns):
    masque_coordonnees = (
        restaurants["LATITUDE"].between(48.0, 49.3)
        & restaurants["LONGITUDE"].between(1.4, 3.7)
    )

    print("Restaurants avec coordonnées plausibles :", masque_coordonnees.sum())
    print("Coordonnées absentes ou à contrôler :", (~masque_coordonnees).sum())

Restaurants avec coordonnées plausibles : 47856
Coordonnées absentes ou à contrôler : 1313


In [24]:
#Créer la table géolocalisée des restaurants

colonnes_restaurants = [
    "CLE_EQUIPEMENT",
    "SIRET",
    "CODGEO",
    "DEPCOM",
    "NOMRS",
    "CNOMRS",
    "NUMVOIE",
    "TYPVOIE",
    "LIBVOIE",
    "CODPOS",
    "TYPEQU",
    "APET",
    "TYPERESTO",
    "CATEGORIE_RESTAURATION",
    "LATITUDE",
    "LONGITUDE",
]

colonnes_restaurants = [
    colonne
    for colonne in colonnes_restaurants
    if colonne in restaurants.columns
]

restaurants_final = restaurants[colonnes_restaurants].copy()

FICHIER_RESTAURANTS = (
    DOSSIER_PROCESSED / "restaurants_idf_2025.csv"
)

restaurants_final.to_csv(
    FICHIER_RESTAURANTS,
    index=False,
    encoding="utf-8-sig",
)

print("Fichier créé :", FICHIER_RESTAURANTS)
print("Nombre de restaurants :", len(restaurants_final))

Fichier créé : C:\Users\almou\OneDrive\Geomarketing_IDF\data\processed\restaurants_idf_2025.csv
Nombre de restaurants : 49169


In [26]:
#Agréger par commune

categories_attendues = [
    "RAPIDE",
    "TRADITIONNELLE",
    "CAFETERIA_LIBRE_SERVICE",
    "TYPE_INCONNU",
]

table_categories = pd.crosstab(
    restaurants["CODGEO"],
    restaurants["CATEGORIE_RESTAURATION"],
)

table_categories = table_categories.reindex(
    columns=categories_attendues,
    fill_value=0,
)

table_categories = table_categories.rename(
    columns={
        "RAPIDE": "NB_RESTAURATION_RAPIDE",
        "TRADITIONNELLE": "NB_RESTAURATION_TRADITIONNELLE",
        "CAFETERIA_LIBRE_SERVICE": "NB_CAFETERIAS_LIBRE_SERVICE",
        "TYPE_INCONNU": "NB_RESTAURATION_TYPE_INCONNU",
    }
).reset_index()

totaux = (
    restaurants.groupby("CODGEO")
    .size()
    .rename("NB_RESTAURANTS_TOTAL")
    .reset_index()
)

concurrence = totaux.merge(
    table_categories,
    on="CODGEO",
    how="left",
    validate="one_to_one",
)

colonnes_nombres = [
    "NB_RESTAURANTS_TOTAL",
    "NB_RESTAURATION_RAPIDE",
    "NB_RESTAURATION_TRADITIONNELLE",
    "NB_CAFETERIAS_LIBRE_SERVICE",
    "NB_RESTAURATION_TYPE_INCONNU",
]

concurrence[colonnes_nombres] = (
    concurrence[colonnes_nombres]
    .fillna(0)
    .astype(int)
)

display(concurrence.head())

,CODGEO,NB_RESTAURANTS_TOTAL,NB_RESTAURATION_RAPIDE,NB_RESTAURATION_TRADITIONNELLE,NB_CAFETERIAS_LIBRE_SERVICE,NB_RESTAURATION_TYPE_INCONNU
0,75056,20911,8329,12553,29,0
1,77001,1,0,1,0,0
2,77002,2,2,0,0,0
3,77003,1,0,1,0,0
4,77005,6,2,4,0,0


In [27]:
#Vérifier que les catégories correspondent au total

somme_categories = (
    concurrence["NB_RESTAURATION_RAPIDE"]
    + concurrence["NB_RESTAURATION_TRADITIONNELLE"]
    + concurrence["NB_CAFETERIAS_LIBRE_SERVICE"]
    + concurrence["NB_RESTAURATION_TYPE_INCONNU"]
)

assert (
    concurrence["NB_RESTAURANTS_TOTAL"] == somme_categories
).all()

assert concurrence["CODGEO"].is_unique
assert concurrence["CODGEO"].str.fullmatch(r"\d{5}").all()

print("Agrégation communale cohérente ✅")

Agrégation communale cohérente ✅


In [28]:
#Détecter le colonne de population

colonnes_population_possibles = [
    "POPULATION_2022"
]

COLONNE_POPULATION = next(
    (
        colonne
        for colonne in colonnes_population_possibles
        if colonne in profil.columns
    ),
    None,
)

if COLONNE_POPULATION is None:
    raise ValueError(
        "Colonne de population introuvable dans le profil. "
        f"Colonnes disponibles : {profil.columns.tolist()}"
    )

profil[COLONNE_POPULATION] = pd.to_numeric(
    profil[COLONNE_POPULATION]
    .astype(str)
    .str.replace("\u202f", "", regex=False)
    .str.replace(" ", "", regex=False)
    .str.replace(",", ".", regex=False),
    errors="coerce",
)

print("Colonne de population utilisée :", COLONNE_POPULATION)

Colonne de population utilisée : POPULATION_2022


In [29]:
#Vérifier les communes non appariées

codes_profil = set(profil["CODGEO"].dropna())
codes_concurrence = set(concurrence["CODGEO"].dropna())

codes_non_apparies = sorted(codes_concurrence - codes_profil)

diagnostic_non_apparies = concurrence[
    concurrence["CODGEO"].isin(codes_non_apparies)
].copy()

print("Codes BPE absents du profil :", len(codes_non_apparies))
display(diagnostic_non_apparies.head(20))

FICHIER_DIAGNOSTIC = (
    DOSSIER_PROCESSED / "diagnostic_codes_bpe_non_apparies_j6.csv"
)

diagnostic_non_apparies.to_csv(
    FICHIER_DIAGNOSTIC,
    index=False,
    encoding="utf-8-sig",
)

Codes BPE absents du profil : 0


,CODGEO,NB_RESTAURANTS_TOTAL,NB_RESTAURATION_RAPIDE,NB_RESTAURATION_TRADITIONNELLE,NB_CAFETERIAS_LIBRE_SERVICE,NB_RESTAURATION_TYPE_INCONNU


In [30]:
#Joindre au profil communal

profil_j6 = profil.merge(
    concurrence,
    on="CODGEO",
    how="left",
    validate="one_to_one",
)

profil_j6[colonnes_nombres] = (
    profil_j6[colonnes_nombres]
    .fillna(0)
    .astype(int)
)

print("Communes avant jointure :", len(profil))
print("Communes après jointure :", len(profil_j6))

assert len(profil_j6) == len(profil)
assert profil_j6["CODGEO"].is_unique

print("Jointure réussie ✅")

Communes avant jointure : 1266
Communes après jointure : 1266
Jointure réussie ✅


In [31]:
#Calculer les indicateurs de concurrence

population = profil_j6[COLONNE_POPULATION].replace(0, np.nan)

profil_j6["DENSITE_RESTAURANTS_10000_HAB"] = (
    profil_j6["NB_RESTAURANTS_TOTAL"]
    / population
    * 10_000
)

profil_j6["DENSITE_RESTAURATION_RAPIDE_10000_HAB"] = (
    profil_j6["NB_RESTAURATION_RAPIDE"]
    / population
    * 10_000
)

profil_j6["DENSITE_RESTAURATION_TRAD_10000_HAB"] = (
    profil_j6["NB_RESTAURATION_TRADITIONNELLE"]
    / population
    * 10_000
)

profil_j6["PART_RESTAURATION_RAPIDE_PCT"] = np.where(
    profil_j6["NB_RESTAURANTS_TOTAL"] > 0,
    profil_j6["NB_RESTAURATION_RAPIDE"]
    / profil_j6["NB_RESTAURANTS_TOTAL"]
    * 100,
    np.nan,
)

profil_j6["PART_RESTAURATION_TRADITIONNELLE_PCT"] = np.where(
    profil_j6["NB_RESTAURANTS_TOTAL"] > 0,
    profil_j6["NB_RESTAURATION_TRADITIONNELLE"]
    / profil_j6["NB_RESTAURANTS_TOTAL"]
    * 100,
    np.nan,
)

In [32]:
population_idf = profil_j6[COLONNE_POPULATION].sum()
restaurants_rapides_idf = profil_j6["NB_RESTAURATION_RAPIDE"].sum()

densite_rapide_idf = (
    restaurants_rapides_idf
    / population_idf
    * 10_000
)

profil_j6["INDICE_DENSITE_RAPIDE_IDF_BASE100"] = (
    profil_j6["DENSITE_RESTAURATION_RAPIDE_10000_HAB"]
    / densite_rapide_idf
    * 100
)

print(
    "Densité régionale de restauration rapide :",
    round(densite_rapide_idf, 2),
    "pour 10 000 habitants",
)

Densité régionale de restauration rapide : 19.64 pour 10 000 habitants


In [33]:
profil_j6

,CODGEO,NOM_COMMUNE,DEP,REG,POPULATION_2011,POPULATION_2016,POPULATION_2022,POP_0_14_ANS_2022,POP_15_29_ANS_2022,POP_30_44_ANS_2022,POP_45_59_ANS_2022,POP_60_74_ANS_2022,POP_75_89_ANS_2022,POP_90_ANS_PLUS_2022,POP_MOINS_30_ANS_2022,POP_15_44_ANS_2022,POP_60_ANS_PLUS_2022,POP_75_ANS_PLUS_2022,PART_0_14_ANS_PCT,PART_15_29_ANS_PCT,PART_30_44_ANS_PCT,PART_MOINS_30_ANS_PCT,PART_15_44_ANS_PCT,PART_60_ANS_PLUS_PCT,PART_75_ANS_PLUS_PCT,EVOLUTION_POP_2016_2022,EVOLUTION_POP_2016_2022_PCT,TAUX_ANNUEL_POP_2016_2022_PCT,REVENU_MEDIAN_EUROS,TAUX_PAUVRETE_PCT,EMPLOIS_SALARIES_LIEU_TRAVAIL,NOMBRE_ETABLISSEMENTS,EMPLOIS_SALARIES_POUR_100_HAB,ETABLISSEMENTS_POUR_1000_HAB,NB_RESTAURANTS_TOTAL,NB_RESTAURATION_RAPIDE,NB_RESTAURATION_TRADITIONNELLE,NB_CAFETERIAS_LIBRE_SERVICE,NB_RESTAURATION_TYPE_INCONNU,DENSITE_RESTAURANTS_10000_HAB,DENSITE_RESTAURATION_RAPIDE_10000_HAB,DENSITE_RESTAURATION_TRAD_10000_HAB,PART_RESTAURATION_RAPIDE_PCT,PART_RESTAURATION_TRADITIONNELLE_PCT,INDICE_DENSITE_RAPIDE_IDF_BASE100
0,75056,Paris,75,11,2249975.0,2190327.0,2113705.0,274280.561508206,513386.75602884,457511.658563684,387099.215350194,302819.494109117,154040.332161959,24566.9822779995,787667.317537046,970898.4145925241,481426.8085490755,178607.31443995849,12.98,24.29,21.65,37.26,45.93,22.78,8.45,-76622.0,-3.5,-0.59,33650.0,16.8,1551133.75022,188002.0,73.38459010221389,88.94429449710343,20911,8329,12553,29,0,98.930551,39.404742,59.388609,39.830711,60.030606,200.612151
1,77001,Achères-la-Forêt,77,11,1232.0,1139.0,1183.0,182.135094788161,159.522993092748,189.869837113436,347.331906368833,204.113784168138,89.442063221765,10.5843212469189,341.658087880909,349.39283020618404,304.1401686368219,100.0263844686839,15.4,13.48,16.05,28.88,29.53,25.71,8.46,44.0,3.86,0.63,34840.0,NaN,130.30679,35.0,11.014944209636518,29.585798816568047,1,0,1,0,0,8.453085,0.000000,8.453085,0.000000,100.000000,0.000000
2,77002,Amillis,77,11,781.0,819.0,821.0,127.936213516012,122.351987081097,146.913593456989,151.60336129515,188.399316742963,59.3206016148823,24.4749262929075,250.28820059710898,269.265580538086,272.1948446507528,83.7955279077898,15.58,14.9,17.89,30.49,32.8,33.15,10.21,2.0,0.24,0.04,29130.0,NaN,190.69785,39.0,23.227509135200975,47.503045066991476,2,2,0,0,0,24.360536,24.360536,0.000000,100.000000,0.000000,124.021102
3,77003,Amponville,77,11,388.0,351.0,351.0,63.0,38.0,72.0,88.0,62.0,28.0,0.0,101.0,110.0,90.0,28.0,17.95,10.83,20.51,28.77,31.34,25.64,7.98,0.0,0.0,0.0,30310.0,NaN,19.95946,12.0,5.6864558404558405,34.18803418803419,1,0,1,0,0,28.490028,0.000000,28.490028,0.000000,100.000000,0.000000
4,77004,Andrezel,77,11,299.0,285.0,324.0,77.0,33.0,78.0,53.0,67.0,15.0,1.0,110.0,111.0,83.0,16.0,23.77,10.19,24.07,33.95,34.26,25.62,4.94,39.0,13.68,2.16,31970.0,NaN,23.16751,13.0,7.150466049382716,40.123456790123456,0,0,0,0,0,0.000000,0.000000,0.000000,NaN,NaN,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1261,95676,Villers-en-Arthies,95,11,470.0,506.0,487.0,102.375958040869,65.2411619107917,95.2581136452168,113.056188915392,78.3083947759905,31.7224914961791,1.03769121556176,167.6171199516607,160.4992755560085,111.06857748773136,32.76018271174086,21.02,13.4,19.56,34.42,32.96,22.81,6.73,-18.999999999998977,-3.75,-0.64,30180.0,NaN,66.81794,13.0,13.720316221765883,26.69404517453793,1,1,0,0,0,20.533881,20.533881,0.000000,100.000000,0.000000,104.539348
1262,95678,Villiers-Adam,95,11,828.0,863.0,848.0,148.0,121.0,167.0,215.0,113.0,79.0,5.0,269.0,288.0,197.0,84.0,17.45,14.27,19.69,31.72,33.96,23.23,9.91,-15.0,-1.74,-0.29,34300.0,NaN,52.62876,26.0,6.2062216981132075,30.660377358490567,1,0,1,0,0,11.792453,0.000000,11.792453,0.000000,100.000000,0.000000
1263,95680,Villiers-le-Bel,95,11,27428.0,27247.0,29238.0,7664.09981092915,6047.91069669275,6202.78529320202,4900.46718294678,3263.78781678669,990.1434769518,168.805722490822,13712.0105076219,12250.69598989477

In [34]:
#Controler Saint-Denis et Pierrefitte

colonnes_controle = [
    "CODGEO",
    COLONNE_POPULATION,
    "NB_RESTAURANTS_TOTAL",
    "NB_RESTAURATION_RAPIDE",
    "NB_RESTAURATION_TRADITIONNELLE",
    "DENSITE_RESTAURATION_RAPIDE_10000_HAB",
]

colonnes_controle = [
    colonne
    for colonne in colonnes_controle
    if colonne in profil_j6.columns
]

controle_saint_denis = profil_j6[
    profil_j6["CODGEO"].isin(["93066", "93059"])
][colonnes_controle]

display(controle_saint_denis)

assert "93066" in profil_j6["CODGEO"].values

if "93059" in profil_j6["CODGEO"].values:
    print(
        "Attention : l'ancien code de Pierrefitte-sur-Seine est encore présent."
    )
else:
    print(
        "Saint-Denis est présente sous le code 93066 "
        "et Pierrefitte n'apparaît plus séparément ✅"
    )

,CODGEO,POPULATION_2022,NB_RESTAURANTS_TOTAL,NB_RESTAURATION_RAPIDE,NB_RESTAURATION_TRADITIONNELLE,DENSITE_RESTAURATION_RAPIDE_10000_HAB
1027,93066,148907.0,598,356,242,23.90754


Saint-Denis est présente sous le code 93066 et Pierrefitte n'apparaît plus séparément ✅


In [35]:
#Afficher les communes les plus concurrentielles

colonnes_classement = [
    "CODGEO",
    "LIBELLE",
    "NOM_COMMUNE",
    COLONNE_POPULATION,
    "NB_RESTAURANTS_TOTAL",
    "NB_RESTAURATION_RAPIDE",
    "DENSITE_RESTAURATION_RAPIDE_10000_HAB",
    "INDICE_DENSITE_RAPIDE_IDF_BASE100",
]

colonnes_classement = [
    colonne
    for colonne in colonnes_classement
    if colonne in profil_j6.columns
]

classement_nombre = profil_j6.sort_values(
    "NB_RESTAURATION_RAPIDE",
    ascending=False,
)[colonnes_classement].head(20)

classement_densite = (
    profil_j6[
        profil_j6[COLONNE_POPULATION] >= 5_000
    ]
    .sort_values(
        "DENSITE_RESTAURATION_RAPIDE_10000_HAB",
        ascending=False,
    )[colonnes_classement]
    .head(20)
)

print("Plus grand nombre de restaurants rapides :")
display(classement_nombre)
print("\n")
print("Plus forte densité, parmi les communes de 5 000 habitants ou plus :")
display(classement_densite)

Plus grand nombre de restaurants rapides :


,CODGEO,NOM_COMMUNE,POPULATION_2022,NB_RESTAURANTS_TOTAL,NB_RESTAURATION_RAPIDE,DENSITE_RESTAURATION_RAPIDE_10000_HAB,INDICE_DENSITE_RAPIDE_IDF_BASE100
0,75056,Paris,2113705.0,20911,8329,39.404742,200.612151
1027,93066,Saint-Denis,148907.0,598,356,23.907540,121.714868
1016,93048,Montreuil,110758.0,534,303,27.356940,139.275996
965,92012,Boulogne-Billancourt,120205.0,564,253,21.047377,107.153593
1088,95018,Argenteuil,107135.0,330,247,23.055024,117.374655
997,93001,Aubervilliers,89489.0,368,244,27.265921,138.812610
979,92044,Levallois-Perret,68412.0,463,197,28.796118,146.602942
1053,94041,Ivry-sur-Seine,64526.0,345,196,30.375353,154.642932
987,92062,Puteaux,44198.0,335,188,42.535861,216.552888
1048,94028,Créteil,92859.0,283,185,19.922678,101.427676




Plus forte densité, parmi les communes de 5 000 habitants ou plus :


,CODGEO,NOM_COMMUNE,POPULATION_2022,NB_RESTAURANTS_TOTAL,NB_RESTAURATION_RAPIDE,DENSITE_RESTAURATION_RAPIDE_10000_HAB,INDICE_DENSITE_RAPIDE_IDF_BASE100
428,77449,Serris,9988.0,88,48,48.057669,244.664777
987,92062,Puteaux,44198.0,335,188,42.535861,216.552888
0,75056,Paris,2113705.0,20911,8329,39.404742,200.612151
112,77118,Claye-Souilly,12381.0,67,46,37.153703,189.151964
662,78401,Meulan-en-Yvelines,8996.0,48,31,34.459760,175.436920
239,77251,Lieusaint,14096.0,82,46,32.633371,166.138654
1022,93057,Les Pavillons-sous-Bois,24872.0,134,81,32.566742,165.799439
1028,93070,Saint-Ouen-sur-Seine,53041.0,322,169,31.862144,162.212288
1053,94041,Ivry-sur-Seine,64526.0,345,196,30.375353,154.642932
1068,94065,Rungis,5669.0,47,17,29.987652,152.669124


In [36]:
#Contrôles finaux

assert len(profil_j6) == len(profil)
assert profil_j6["CODGEO"].is_unique
assert profil_j6["CODGEO"].str.fullmatch(r"\d{5}").all()

for colonne in colonnes_nombres:
    assert profil_j6[colonne].ge(0).all()

assert (
    profil_j6["NB_RESTAURATION_RAPIDE"]
    <= profil_j6["NB_RESTAURANTS_TOTAL"]
).all()

valeurs_part = profil_j6["PART_RESTAURATION_RAPIDE_PCT"].dropna()
assert valeurs_part.between(0, 100).all()

print("Tous les contrôles J6 sont validés ✅")

Tous les contrôles J6 sont validés ✅


In [37]:
#Enregistrer les fichiers


FICHIER_CONCURRENCE = (
    DOSSIER_PROCESSED / "concurrence_restauration_idf_2025.csv"
)

FICHIER_PROFIL_J6 = (
    DOSSIER_PROCESSED / "profil_communes_idf_j6.csv"
)

concurrence.to_csv(
    FICHIER_CONCURRENCE,
    index=False,
    encoding="utf-8-sig",
)

profil_j6.to_csv(
    FICHIER_PROFIL_J6,
    index=False,
    encoding="utf-8-sig",
)

print("Fichier communal concurrence :", FICHIER_CONCURRENCE)
print("Profil enrichi :", FICHIER_PROFIL_J6)
print("Restaurants géolocalisés :", FICHIER_RESTAURANTS)
print("J6 terminé ✅")

Fichier communal concurrence : C:\Users\almou\OneDrive\Geomarketing_IDF\data\processed\concurrence_restauration_idf_2025.csv
Profil enrichi : C:\Users\almou\OneDrive\Geomarketing_IDF\data\processed\profil_communes_idf_j6.csv
Restaurants géolocalisés : C:\Users\almou\OneDrive\Geomarketing_IDF\data\processed\restaurants_idf_2025.csv
J6 terminé ✅
